# 🧪 Lab 06: What Catalyst Cannot See

Welcome to the optimizer-evidence autopsy bay. In this lab, two expressions return the same rows, but only one exposes its logic as a native Catalyst expression tree.

**Mission Objective:** write a small Parquet dataset and compare a native predicate with the same rule hidden inside a Python UDF. We inspect optimized and physical plans, focusing on the filters Spark can push toward the Parquet scan.

**Deterministic Guardrail:** both versions use the same dataset and the same rule: keep rows where `amount > 50`. This is a plan-visibility experiment, not a runtime benchmark.


### Step 1: Define the diagnostic session
Adaptive execution is disabled so the scan and filtering evidence remain easy to compare. The dataset is written to a temporary local Parquet path and removed before the lab finishes.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType
import shutil

spark = (SparkSession.builder
    .master("local[2]")
    .appName("lab-06-hide-the-evidence")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
path = "/tmp/lab-06-catalyst-visibility-parquet"
shutil.rmtree(path, ignore_errors=True)
print("Spark version:", spark.version)
print("Parquet path:", path)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 06:37:35 WARN Utils: Your hostname, T14-PF4WM3XL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/24 06:37:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/angelalvarez/.local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/24 06:37:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0
Parquet path: /tmp/lab-06-catalyst-visibility-parquet


### Step 2: Create the same evidence for both queries
The rows contain an amount, a category, and a payload column. The payload is intentionally not selected later, so column pruning can also be observed in the scan.


In [2]:
source = (spark.range(0, 1_000)
    .select(
        F.col("id"),
        (F.col("id") % 100).cast("long").alias("amount"),
        F.when((F.col("id") % 2) == 0, F.lit("even")).otherwise(F.lit("odd")).alias("category"),
        F.repeat(F.lit("payload"), 20).alias("payload")
    ))
source.write.mode("overwrite").parquet(path)
dataset = spark.read.parquet(path)
print("Rows written:", dataset.count())
print("Dataset schema:")
dataset.printSchema()


Rows written: 1000
Dataset schema:
root
 |-- id: long (nullable = true)
 |-- amount: long (nullable = true)
 |-- category: string (nullable = true)
 |-- payload: string (nullable = true)



### Step 3: Keep the rule visible to Catalyst
The native comparison is represented directly as a Catalyst expression. Inspect the optimized and physical plans for `PushedFilters`, especially `GreaterThan(amount,50)`.


In [3]:
native_query = (dataset
    .where(F.col("amount") > 50)
    .select("id", "amount", "category"))

print("Native result count: deferred until comparison")
print("=== Native predicate: extended plan ===")
native_query.explain("extended")
print("=== Native predicate: formatted plan ===")
native_query.explain("formatted")


Native result count: deferred until comparison
=== Native predicate: extended plan ===
== Parsed Logical Plan ==
'Project ['id, 'amount, 'category]
+- Filter (amount#5L > cast(50 as bigint))
   +- Relation [id#4L,amount#5L,category#6,payload#7] parquet

== Analyzed Logical Plan ==
id: bigint, amount: bigint, category: string
Project [id#4L, amount#5L, category#6]
+- Filter (amount#5L > cast(50 as bigint))
   +- Relation [id#4L,amount#5L,category#6,payload#7] parquet

== Optimized Logical Plan ==
Project [id#4L, amount#5L, category#6]
+- Filter (isnotnull(amount#5L) AND (amount#5L > 50))
   +- Relation [id#4L,amount#5L,category#6,payload#7] parquet

== Physical Plan ==
*(1) Filter (isnotnull(amount#5L) AND (amount#5L > 50))
+- *(1) ColumnarToRow
   +- FileScan parquet [id#4L,amount#5L,category#6] Batched: true, DataFilters: [isnotnull(amount#5L), (amount#5L > 50)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/lab-06-catalyst-visibility-parquet], PartitionFilters: [], 

### Step 4: Seal the same rule inside a UDF
The UDF returns the same boolean decision, but the function body is no longer a native Catalyst expression tree. Spark can see the UDF boundary and its return type; it cannot rewrite the Python algorithm as a Parquet predicate.


In [4]:
@F.udf(returnType=BooleanType())
def amount_above_50(value):
    return value is not None and value > 50

udf_query = (dataset
    .where(amount_above_50(F.col("amount")))
    .select("id", "amount", "category"))

print("UDF result count: plan-only in this lab")
print("=== UDF predicate: extended plan ===")
udf_query.explain("extended")
print("=== UDF predicate: formatted plan ===")
udf_query.explain("formatted")


UDF result count: plan-only in this lab
=== UDF predicate: extended plan ===


== Parsed Logical Plan ==
'Project ['id, 'amount, 'category]
+- Filter amount_above_50(amount#5L)#17
   +- Relation [id#4L,amount#5L,category#6,payload#7] parquet

== Analyzed Logical Plan ==
id: bigint, amount: bigint, category: string
Project [id#4L, amount#5L, category#6]
+- Filter amount_above_50(amount#5L)#17
   +- Relation [id#4L,amount#5L,category#6,payload#7] parquet

== Optimized Logical Plan ==
Project [id#4L, amount#5L, category#6]
+- Filter pythonUDF0#18: boolean
   +- ArrowEvalPython [amount_above_50(amount#5L)#17], [pythonUDF0#18], 101
      +- Project [id#4L, amount#5L, category#6]
         +- Relation [id#4L,amount#5L,category#6,payload#7] parquet

== Physical Plan ==
*(1) Project [id#4L, amount#5L, category#6]
+- *(1) Filter pythonUDF0#18: boolean
   +- ArrowEvalPython [amount_above_50(amount#5L)#17], [pythonUDF0#18], 101
      +- FileScan parquet [id#4L,amount#5L,category#6] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tm

/home/angelalvarez/.local/lib/python3.12/site-packages/pyspark/sql/udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


### Step 5: Compare the evidence directly
We execute both queries to confirm that the result counts match. The plans should not tell the same story: the native scan can expose the amount predicate to Parquet, while the UDF plan introduces Python evaluation and cannot push the hidden comparison as a native data-source filter.


In [5]:
native_count = native_query.count()
print("Native count:", native_count)
print("UDF count: not materialized; plan comparison is the experiment")
native_plan = native_query._jdf.queryExecution().executedPlan().toString()
udf_plan = udf_query._jdf.queryExecution().executedPlan().toString()
print("Native plan exposes GreaterThan pushdown:", "GreaterThan(amount" in native_plan)
print("UDF plan contains Python evaluation:", "Python" in udf_plan)
shutil.rmtree(path, ignore_errors=True)


Native count: 490
UDF count: not materialized; plan comparison is the experiment
Native plan exposes GreaterThan pushdown: True
UDF plan contains Python evaluation: True


# 📊 Post-Lab Analysis: What Catalyst Cannot See

This lab used two equivalent rules and held the data constant. The native expression and the Python UDF returned the same count, but they exposed different information to Catalyst and the Parquet reader.

### 1. Native Expressions Carry Structure

The native predicate appears as a comparison in the optimized plan and can be represented in the scan's pushed-filter metadata. Spark can reason about its operator, input column, type, and null behavior.

### 2. The UDF Creates a Sealed Envelope

The UDF plan introduces a Python evaluation boundary. Spark knows that a function consumes `amount` and returns a boolean, but the comparison inside the Python body is not available as a native Parquet predicate. The logic became less visible before row execution began.

### 3. Same Result, Different Opportunity

The matching counts show that the two forms are semantically equivalent for this data. They do not provide Catalyst with equivalent optimization opportunities. The native version can expose the predicate as a data-source filter; the UDF version cannot expose the hidden comparison in the same way.

The lesson is not that UDFs are forbidden. It is that **being executable is not the same as being visible**. Catalyst cannot act on evidence it was never allowed to see.
